# Course 1: Geometric Deep Learning and Graphs
## Session 5: Graph transformers

**Goal:** compute Laplacian and random-walk positional encodings, and show a task that message passing fails without them.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. Two positional encodings
- Laplacian PE: the eigenvectors of the graph Laplacian with the smallest non-zero eigenvalues.
- Random-walk PE: for each node, the chance that a random walk returns to it after 1, 2, ..., k steps.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

def cycle(n):
    A = np.zeros((n, n), dtype=np.float32)
    for i in range(n): A[i, (i + 1) % n] = A[(i + 1) % n, i] = 1
    return A

def laplacian_pe(A, k):
    d = A.sum(1); L = np.eye(len(A)) - A / np.sqrt(np.outer(d, d))
    vals, vecs = np.linalg.eigh(L)
    return vecs[:, 1:k + 1]                       # skip the constant eigenvector; signs are arbitrary

def rw_pe(A, k):
    P = A / A.sum(1, keepdims=True); M = np.eye(len(A)); out = []
    for _ in range(k):
        M = M @ P; out.append(np.diag(M))
    return np.stack(out, 1).astype(np.float32)

print("Laplacian PE of a 6-cycle (2 columns):"); print(laplacian_pe(cycle(6), 2).round(2))
print("random-walk PE, 6-cycle node 0  :", rw_pe(cycle(6), 4)[0].round(3))
print("random-walk PE, triangle node 0 :", rw_pe(cycle(3), 4)[0].round(3), " <- returns after 3 steps")

### 2. A task: one big ring, or many small rings?
Every node has degree 2 and there are no features, so a plain message-passing network sees the same thing at every node of every graph.

In [ ]:
def rings(n_total, ring_len):
    """n_total nodes arranged as separate rings of length ring_len (ring_len = n_total gives one big ring)."""
    A = np.zeros((n_total, n_total), dtype=np.float32)
    for s in range(0, n_total, ring_len): A[s:s + ring_len, s:s + ring_len] = cycle(ring_len)
    return A

train = [(rings(n, r), int(r < n)) for n in (12, 24) for r in (3, 4, n)]
test = [(rings(n, r), int(r < n)) for n in (36, 60) for r in (3, 4, n)]        # larger graphs than any seen in training

class GIN(nn.Module):
    def __init__(self, d_in, d=32, layers=3):
        super().__init__()
        self.mlps = nn.ModuleList([nn.Sequential(nn.Linear(d_in if i == 0 else d, d), nn.ReLU(), nn.Linear(d, d)) for i in range(layers)])
        self.out = nn.Linear(d, 1)
    def forward(self, A, X):
        for mlp in self.mlps: X = torch.relu(mlp(X + A @ X))
        return self.out(X.mean(0))            # mean readout, so graph size alone gives nothing away

def run(use_pe, k=6, seed=0):
    torch.manual_seed(seed)
    feats = lambda A: torch.tensor(rw_pe(A, k)) if use_pe else torch.ones(len(A), 1)
    model = GIN(k if use_pe else 1); opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for epoch in range(200):
        for A, t in train:
            loss = F.binary_cross_entropy_with_logits(model(torch.tensor(A), feats(A)), torch.tensor([float(t)]))
            opt.zero_grad(); loss.backward(); opt.step()
    acc = lambda data: np.mean([(model(torch.tensor(A), feats(A)).item() > 0) == bool(t) for A, t in data])
    return acc(train), acc(test)

plain, with_pe = run(False), run(True)
print("GIN, no positional encoding : train %.2f  test %.2f" % plain)
print("GIN + random-walk PE        : train %.2f  test %.2f" % with_pe)
assert plain[1] <= 0.67 and with_pe[1] == 1.0

Without a positional encoding every node in every graph gets the same embedding, so the network gives one answer for all graphs and can only match the majority class. The positional encoding fixes that before any attention is added.

### 3. The GraphGPS layer in outline
One GPS layer adds a local message-passing step and a global attention step, then applies an MLP.

In [ ]:
class GPSLayer(nn.Module):
    def __init__(self, d, heads=4):
        super().__init__()
        self.local = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, d))
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 2 * d), nn.ReLU(), nn.Linear(2 * d, d))
        self.n1, self.n2, self.n3 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
    def forward(self, A, X):
        h_local = self.n1(X + self.local(X + A @ X))             # neighbours only
        h_global = self.n2(X + self.attn(X[None], X[None], X[None])[0][0])   # every node attends to every node
        h = h_local + h_global
        return self.n3(h + self.ff(h))

A = torch.tensor(rings(12, 12)); X = nn.Linear(8, 32)(torch.tensor(rw_pe(A.numpy(), 8)))
print("GPS layer output:", GPSLayer(32)(A, X).shape)

### 4. GIN against GPS on ZINC (NOT RUN here)

In [ ]:
# NOT RUN: needs torch_geometric, a download of the ZINC subset (12k molecules) and ideally a GPU.
import torch_geometric.transforms as T
from torch_geometric.datasets import ZINC
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, GPSConv, global_add_pool

tf = T.AddRandomWalkPE(walk_length=20, attr_name="pe")
train_ds, val_ds = ZINC("data/zinc", subset=True, split="train", pre_transform=tf), ZINC("data/zinc", subset=True, split="val", pre_transform=tf)
# Build two models with about the same number of parameters (the benchmark budget is 500k):
#   A: 4 x GINEConv                      B: 4 x GPSConv(channels, GINEConv(...), heads=4)
# Train both with L1 loss and report validation MAE over 3 seeds.
# The GraphGPS paper reports about 0.07 test MAE on ZINC; a plain GIN is near 0.39 in the benchmark paper. Check both numbers in the papers.

### TODO
1. Flip the sign of one Laplacian eigenvector. The graph is the same but the encoding changed. How do graph transformers cope with this?
2. In section 2, what is the smallest `k` for which random-walk PE still solves the task? Explain using the ring lengths 3 and 4.
3. Run the ZINC comparison and report MAE with the parameter counts.